# Set up

In [ ]:
!git clone https://github.com/NLP-Reichman/2025_assignment_1.git
!mv 2025_assignment_1/data data
!rm 2025_assignment_1/ -r

Cloning into '2025_assignment_1'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 71 (delta 24), reused 44 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 7.05 MiB | 17.96 MiB/s, done.
Resolving deltas: 100% (24/24), done.
mv: cannot move '2025_assignment_1/data' to 'data/data': Directory not empty


# Introduction
In this assignment you will be creating tools for learning and testing language models. The corpora that you will be working with are lists of tweets in 8 different languages that use the Latin script. The data is provided either formatted as CSV or as JSON, for your convenience. The end goal is to write a set of tools that can detect the language of a given tweet.
The relevant files are under the data folder:

- en.csv (or the equivalent JSON file)
- es.csv (or the equivalent JSON file)
- fr.csv (or the equivalent JSON file)
- in.csv (or the equivalent JSON file)
- it.csv (or the equivalent JSON file)
- nl.csv (or the equivalent JSON file)
- pt.csv (or the equivalent JSON file)
- tl.csv (or the equivalent JSON file)

In [ ]:
import os
import json
from google.colab import files
import pandas as pd
import numpy as np
from itertools import product
import csv
from collections import defaultdict
import math


# Implementation

## Part 1
Implement the function *preprocess* that iterates over all the data files and creates a single vocabulary, containing all the tokens in the data. Our token definition is a single UTF-8 encoded character. So, the vocabulary list is a simple Python list of all the characters that you see at least once in the data. The vocabulary should include the `<start>` and  `<end>` tokens.

Note - do NOT lowecase the sentences.

In [ ]:

# def preprocess() -> list[str]:
#     '''
#     Return a list of characters, representing the shared vocabulary of all languages
#     '''
#     vocab_set = set()

#     data_folder = 'data'
#     for filename in os.listdir(data_folder):
#         if filename.endswith('.csv'):
#             file_path = os.path.join(data_folder, filename)
#             with open(file_path, 'r', encoding='utf-8') as csvfile:
#                 reader = csv.reader(csvfile)
#                 for row in reader:
#                     for cell in row:
#                         for char in cell:
#                             vocab_set.add(char)

#     # Add special tokens
#     vocab = sorted(vocab_set)
#     vocab.insert(0, '<start>')
#     vocab.insert(1, '<end>')

#     return vocab

In [ ]:
def preprocess() -> list[str]:
  '''
  Return a list of characters, representing the shared vocabulary of all languages
  '''
  vocab = []
  ########################################
  # TODO: your code
  data_dir = 'data'
  for filename in os.listdir(data_dir):
      file_path = os.path.join(data_dir, filename)
      if os.path.isfile(file_path):
          with open(file_path, 'r', encoding='utf-8') as f:
              reader = csv.DictReader(f)
              for row in reader:
                  text = row.get("tweet_text", "").strip()
                  for char in text:
                     if char not in vocab:
                        vocab.append(char)

  vocab = ['<start>', '<end>'] + vocab

  ########################################
  return vocab

In [ ]:
vocab = preprocess()
print(f"vocab length: {len(vocab)}")
print(f"Some characters in the vocab: {vocab[:10]}")

vocab length: 1804
Some characters in the vocab: ['<start>', '<end>', '@', 's', 'h', 'a', 'n', 'o', 'c', 't']


## Part 2
Implement the function *build_lm* that generates a language model from a textual corpus. The function should return a dictionary (representing a model) where the keys are all the relevant *n*-1 sequences, and the values are dictionaries with the *n*_th tokens and their corresponding probabilities to occur. To ensure consistent probabilities calculation, please add n-1 `<start>` tokens to the beginning of a tweet and one `<end>` token at the end. For example, for a trigram model (tokens are characters), it should look something like:

{ "ab":{"c":0.5, "b":0.25, "d":0.25}, "ca":{"a":0.2, "b":0.7, "d":0.1} }

which means for example that after the sequence "ab", there is a 0.5 chance that "c" will appear, 0.25 for "b" to appear and 0.25 for "d" to appear.

Note - You should think how to add the add_one smoothing information to the dictionary and implement it.

Please add the `<unk>` token with $p(<unk>)=1|V|$ to the LM if buiulding a smoothed LM.

In [ ]:
# def build_lm(lang: str, n: int, smoothed: bool = False) -> dict[str, dict[str, float]]:
#     '''
#     Return a language model for the given lang and n_gram (n)
#     :param lang: the language of the model
#     :param n: the n_gram value
#     :param smoothed: boolean indicating whether to apply smoothing
#     :return: a dictionary where the keys are n_grams and the values are dictionaries
#     '''
#     LM = defaultdict(lambda: defaultdict(int))
#     data_path = os.path.join('data', f'{lang}.csv')

#     # Get the shared vocabulary (from all languages)
#     vocab = preprocess()
#     if smoothed and '<unk>' not in vocab:
#         vocab.append('<unk>')

#     V = len(vocab)

#     # First pass: Count n-grams
#     with open(data_path, 'r', encoding='utf-8') as csvfile:
#         reader = csv.reader(csvfile)
#         for row in reader:
#             for sentence in row:
#                 tokens = ['<start>'] * (n - 1) + list(sentence) + ['<end>']
#                 for i in range(len(tokens) - n + 1):
#                     context = ''.join(tokens[i:i + n - 1])
#                     next_char = tokens[i + n - 1]
#                     LM[context][next_char] += 1

#     # Second pass: Normalize with optional smoothing
#     LM_final = {}
#     for context, next_chars in LM.items():
#         LM_final[context] = {}

#         if smoothed:
#             # Add 1 to all characters in vocab (including <unk>) to apply add-one smoothing
#             for char in vocab:
#                 if char not in next_chars:
#                     next_chars[char] = 1
#                 else:
#                     next_chars[char] += 1
#         else:
#             # If not smoothed and <unk> is missing, do nothing
#             pass

#         total = sum(next_chars.values())

#         # Now compute the normalized probabilities
#         for char in vocab:
#             count = next_chars.get(char, 0)
#             LM_final[context][char] = count / total

#     return LM_final

In [ ]:
def build_lm(lang: str, n: int, smoothed: bool = False) -> dict[str, dict[str, float]]:
  '''
  Return a language model for the given lang and n_gram (n)
  :param lang: the language of the model
  :param n: the n_gram value
  :param smoothed: boolean indicating whether to apply smoothing
  :return: a dictionary where the keys are n_grams and the values are dictionaries
  '''
  LM = {}
  ########################################
  LM = defaultdict(lambda: defaultdict(int))

  with open(f"data/{lang}.csv", encoding='utf-8') as f:
      for row in csv.DictReader(f):
          tokens = ['<start>'] * (n - 1) + list(row['tweet_text']) + ['<end>']
          for i in range(len(tokens) - n + 1):
              context = ''.join(tokens[i:i + n - 1])
              LM[context][tokens[i + n - 1]] += 1

  if not smoothed:
      for context, next_chars in LM.items():
          total = sum(next_chars.values())
          LM[context] = {char: count / total for char, count in next_chars.items()}
  else:
      vocab = set(char for next_chars in LM.values() for char in next_chars)
      V = len(vocab) + 1  # +1 for <unk>
      vocab.add('<unk>')
      LM['<unk>'] = {char: 1 / V for char in vocab}
      for context, next_chars in LM.items():
          total = sum(next_chars.values()) + V
          LM[context] = {char: (next_chars.get(char, 0) + 1) / total for char in vocab}
  ########################################
  return LM

In [ ]:
LM = build_lm("en", 3, False)
print(f"English Language Model with 3-gram is of length: {len(LM)}")

English Language Model with 3-gram is of length: 8238


## Part 3
Implement the function *eval* that returns the perplexity of a model (dictionary) running over the data file of the given target language.

The `<unk>` should be used for unknown contexts when calculating the perplexities.

In [ ]:
# def perplexity(model: dict, text: list[str], n: int) -> float:
#     '''
#     Calculates the perplexity of the given string using the given language model.
#     :param model: The language model
#     :param text: The tokenized text to calculate the perplexity for
#     :param n: The n-gram of the model
#     :return: The perplexity
#     '''
#     tokens = ['<start>'] * (n - 1) + text + ['<end>']
#     log_prob = 0.0
#     N = 0

#     for i in range(len(tokens) - n + 1):
#         context = ''.join(tokens[i:i + n - 1])
#         target = tokens[i + n - 1]
#         N += 1

#         if context in model:
#             probs = model[context]
#             prob = probs.get(target, probs.get('<unk>', 1e-8))  # fallback to <unk> or small value
#         else:
#             probs = model.get('<unk>', {})
#             prob = probs.get(target, probs.get('<unk>', 1e-8))  # unknown context

#         log_prob += math.log(prob)

#     pp = math.exp(-log_prob / N) if N > 0 else float('inf')
#     return pp

In [ ]:
# def eval(model: dict, target_lang: str, n: int) -> float:
#     '''
#     Return the perplexity value calculated over applying the model on the text file
#     of the target_lang language.
#     :param model: the language model
#     :param target_lang: the target language
#     :param n: The n-gram of the model
#     :return: the perplexity value
#     '''
#     data_path = os.path.join('data', f'{target_lang}.csv')
#     total_perplexity = 0.0
#     sentence_count = 0

#     with open(data_path, 'r', encoding='utf-8') as csvfile:
#         reader = csv.reader(csvfile)
#         for row in reader:
#             for sentence in row:
#                 tokenized = list(sentence)
#                 pp = perplexity(model, tokenized, n)
#                 total_perplexity += pp
#                 sentence_count += 1

#     avg_pp = total_perplexity / sentence_count if sentence_count > 0 else float('inf')
#     return avg_pp

In [ ]:
def perplexity(model: dict, text: list, n) -> float:
  '''
  Calculates the perplexity of the given string using the given language model.
  :param model: The language model
  :param text: The tokenized text to calculate the perplexity for
  :param n: The n-gram of the model
  :return: The perplexity
  '''
  pp = 0
  #######################################
  # your code
  log_prob_sum = 0.0
  N = 0
  for i in range(len(text) - n + 1):
    context = ''.join(text[i:i + n - 1])
    next_char = text[i + n - 1]
    if context in model:
      prob = model[context].get(next_char, model[context].get('<unk>'))
    else:
      #prob = sum(model['<unk>'].values()) / len(model['<unk>'])
      prob = model['<unk>'].get('<unk>')
    log_prob_sum += math.log(prob)
    N += 1
  pp = math.exp(-log_prob_sum / N) if N > 0 else float('inf')
  ########################################
  return pp

In [ ]:
def eval(model: dict, target_lang: str, n: int) -> float:
  '''
  Return the perplexity value calculated over applying the model on the text file
  of the target_lang language.
  :param model: the language model
  :param target_lang: the target language
  :param n: The n-gram of the model
  :return: the perplexity value
  '''
  pp = 0
  ########################################
  # your code
  N = 0
  total_pp = 0
  with open(f"data/{target_lang}.csv", encoding='utf-8') as f:
      reader = csv.DictReader(f)
      for row in reader:
          tweet = row['tweet_text']
          tokens = ['<start>'] * (n - 1) + list(tweet) + ['<end>']
          total_pp += perplexity(model, tokens, n)
          N += 1

  pp = total_pp / N if N > 0 else float('inf')
  ########################################
  return pp

In [ ]:
LM = build_lm("en", 3, True)

In [ ]:
print("Perplexity of the English 3-gram model on datasets:")
print(f"On English: {eval(LM, 'en', 3): .2f}")
print(f"On French: {eval(LM, 'fr', 3): .2f}")
print(f"On Dutch: {eval(LM, 'nl', 3): .2f}")
print(f"On Tagalog: {eval(LM, 'tl', 3): .2f}")


Perplexity of the English 3-gram model on datasets:
On English:  20.64
On French:  42.75
On Dutch:  48.17
On Tagalog:  54.98


In [ ]:
lm1 = build_lm("en", 1, True)
lm2 = build_lm("en", 2, True)
lm3 = build_lm("en", 3, True)
lm4 = build_lm("en", 4, True)

print("Perplexity on differnet n-gram models on English")
print(f"On 1-gram: {eval(lm1, 'en', 1): .2f}")
print(f"On 2-gram: {eval(lm2, 'en', 2): .2f}")
print(f"On 3-gram: {eval(lm3, 'en', 3): .2f}")
print(f"On 4-gram: {eval(lm4, 'en', 4): .2f}")

Perplexity on differnet n-gram models on English
On 1-gram:  42.68
On 2-gram:  21.05
On 3-gram:  20.64
On 4-gram:  37.21


## Part 4
Implement the *match* function that calls *eval* using a specific value of *n* for every possible language pair among the languages we have data for. You should call *eval* for every language pair four times, with each call assign a different value for *n* (1-4). Each language pair is composed of the source language and the target language. Before you make the call, you need to call the *lm* function to create the language model for the source language. Then you can call *eval* with the language model and the target language. The function should return a pandas DataFrame with the following four columns: *source_lang*, *target_lang*, *n*, *perplexity*. The values for the first two columns are the two-letter language codes. The value for *n* is the *n* you use for generating the specific perplexity values which you should store in the forth column.

In [ ]:
languages = ['en', 'es', 'fr', 'in', 'it', 'nl', 'pt', 'tl']

In [ ]:
# def match() -> dict[str, float]:
#     '''
#     Return a dictionary with keys like "en_en_3" and their perplexity values.
#     '''
#     results = {}

#     for source_lang in languages:
#         for target_lang in languages:
#             for n in range(1, 5):  # n from 1 to 4
#                 print(f"Evaluating {source_lang} model on {target_lang} data with n={n}...")
#                 model = build_lm(source_lang, n, smoothed=True)
#                 pp = eval(model, target_lang, n)
#                 key = f"{source_lang}_{target_lang}_{n}"
#                 results[key] = pp

#     return results

In [ ]:
def match() -> pd.DataFrame:
  '''
  Return a DataFrame containing one line per every language pair and n_gram.
  Each line will contain the perplexity calculated when applying the language model
  of the source language on the text of the target language.
  :return: a DataFrame containing the perplexity values
  '''
  df  = pd.DataFrame()
  ########################################
  # your code
  results = []
  for source_lang in languages:
    for n in range(1, 5):
      model = build_lm(source_lang, n, smoothed=True)
      for target_lang in languages:
        pp = eval(model, target_lang, n)
        results.append({
          'source': source_lang,
          'target': target_lang,
          'n': n,
          'perplexity': pp
        })
  return pd.DataFrame(results)
  ########################################
  return df


## Part 5
Implement the *generate* function which takes a language code, *n*, the prompt (the starting text), the number of tokens to generate, and *r*, which is the random seed for any randomized action you plan to take in your implementation. The function should start generating tokens, one by one, using the language model of the given source language and *n*. The prompt should be used as a starting point for aligning on the probabilities to be used for generating the next token.

Note - The generation of the next token should be from the LM's distribution with NO smoothing.

In [ ]:
# import random

# def generate(lang: str, n: int, prompt: str, number_of_tokens: int, r: int) -> str:
#     '''
#     Generate text in the given language using the given parameters.
#     '''
#     random.seed(r)
#     model = build_lm(lang, n, smoothed=False)

#     generated = list(prompt)

#     for _ in range(number_of_tokens):
#         # Prepare context: last n-1 tokens or pad with <start> if not enough
#         if len(generated) < n - 1:
#             context = ['<start>'] * (n - 1 - len(generated)) + generated
#         else:
#             context = generated[-(n - 1):]

#         context_str = ''.join(context)

#         # Get possible next-token distribution
#         if context_str in model:
#             next_char_probs = model[context_str]
#             chars = list(next_char_probs.keys())
#             probs = list(next_char_probs.values())

#             # Sample next character
#             next_char = random.choices(chars, weights=probs, k=1)[0]
#         else:
#             # No context match — stop early
#             break

#         if next_char == '<end>':
#             break

#         generated.append(next_char)

#     return ''.join(generated)

In [ ]:
import random

def generate(lang: str, n: int, prompt: str, number_of_tokens: int, r: int) -> str:
  '''
  Generate text in the given language using the given parameters.
  :param lang: the language of the model
  :param n: the n_gram value
  :param prompt: the prompt to start the generation
  :param number_of_tokens: the number of tokens to generate
  :param r: the random seed to use
  '''
  text = ""
  ########################################
  # your code
  LM = build_lm(lang, n, smoothed=False)
  random.seed(r)
  generated = list(prompt)
  original_length = len(generated)
  for _ in range(number_of_tokens):
    context = ''.join(generated[-(n - 1):]) if n > 1 else ''
    if context not in LM:
        break  # Stop generation if no known continuation
    next_chars = list(LM[context].keys())
    probs = list(LM[context].values())
    # Sample next character from the learned distribution
    next_char = random.choices(next_chars, weights=probs, k=1)[0]
    if next_char == '<end>':
      break
    generated.append(next_char)
  text = ''.join(generated)
  ########################################
  return text

## Part 6
Play with your generate function, try to generate different texts in different language and various values of *n*. No need to submit anything of that.

In [ ]:
print(generate('en', 1, "I am", 10, 5))
print(generate('en', 2, "I am", 10, 5))
print(generate('en', 3, "I am", 10, 5))
print(generate('en', 4, "I am ", 10, 5))
print(generate('es', 2, "Soy", 10, 5))
print(generate('es', 3, "Soy", 10, 5))
print(generate('fr', 2, "Je suis", 10, 5))
print(generate('fr', 3, "Je suis", 10, 5))

I amtpgLpC eLh
I amoulpeginSh
I amit: Lynmkm
I am carrive Ob
Soycalíodenye
Soy orbe
Je suis:/opapropa
Je suis tune


# Testing

Copy the content of the **tests.py** file from the repo and paste below. This will create the results.json file and download it to your machine.

In [ ]:
########################################
# PLACE TESTS HERE #
# Create tests
def test_preprocess():
    return {
        'vocab_length': len(preprocess()),
    }

def test_build_lm():
    return {
        'english_2_gram_length': len(build_lm('en', 2, True)),
        'english_3_gram_length': len(build_lm('en', 3, True)),
        'french_3_gram_length': len(build_lm('fr', 3, True)),
        'spanish_3_gram_length': len(build_lm('es', 3, True)),
    }

def test_eval():
    lm = build_lm('en', 3, True)
    return {
        'en_on_en': round(eval(lm, 'en', 3), 2),
        'en_on_fr': round(eval(lm, 'fr', 3), 2),
        'en_on_tl': round(eval(lm, 'tl', 3), 2),
        'en_on_nl': round(eval(lm, 'nl', 3), 2),
    }

def test_match():
    df = match()
    return {
        'df_shape': df.shape,
        'en_en_3': df[(df['source'] == 'en') & (df['target'] == 'en') & (df['n'] == 3)]['perplexity'].values[0],
        'en_tl_3': df[(df['source'] == 'en') & (df['target'] == 'tl') & (df['n'] == 3)]['perplexity'].values[0],
        'en_nl_3': df[(df['source'] == 'en') & (df['target'] == 'nl') & (df['n'] == 3)]['perplexity'].values[0],
    }

def test_generate():
    return {
        'english_2_gram': generate('en', 2, "I am", 20, 5),
        'english_3_gram': generate('en', 3, "I am", 20, 5),
        'english_4_gram': generate('en', 4, "I Love", 20, 5),
        'spanish_2_gram': generate('es', 2, "Soy", 20, 5),
        'spanish_3_gram': generate('es', 3, "Soy", 20, 5),
        'french_2_gram': generate('fr', 2, "Je suis", 20, 5),
        'french_3_gram': generate('fr', 3, "Je suis", 20, 5),
    }

TESTS = [test_preprocess, test_build_lm, test_eval, test_match, test_generate]

# Run tests and save results
res = {}
for test in TESTS:
    try:
        cur_res = test()
        res.update({test.__name__: cur_res})
    except Exception as e:
        res.update({test.__name__: repr(e)})

with open('results.json', 'w') as f:
    json.dump(res, f, indent=2)

# Download the results.json file
files.download('results.json')
########################################

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Show the local files, results.json should be there now and
# also downloaded to your local machine
!ls -l

total 12
drwxr-xr-x 3 root root 4096 May  4 07:11 data
-rw-r--r-- 1 root root  848 May  4 07:36 results.json
drwxr-xr-x 1 root root 4096 Apr 30 13:37 sample_data
